#  Collation in SQL Server

> **What this notebook teaches:** Collation is the set of rules SQL Server uses to compare and sort character data — it decides whether `Ali` equals `ali`, how accented characters are handled, and how different alphabets sort correctly. It matters in every database with text columns, and becomes critical the moment a system needs to support more than one language — in this course's case, Persian alongside English. By the end of this notebook, you'll know every collation option, the four levels collation can be set at, and specifically which collation to use for Persian text.

> **Note on runnable code:** Collation is a SQL Server–specific feature with no real equivalent in SQLite, so this notebook uses reference **T-SQL** code cells — written to run in SSMS against a real SQL Server instance, not in this notebook's Python kernel. Where a concept can be demonstrated with SQLite's own (much simpler) collation support, that's marked clearly as an analogy, not the real thing.

## 1. The Problem, Before Any Definitions

Imagine an `Employees` table with three rows for `FirstName`: `Ali`, `ali`, `ALI`. Now run:

```sql
SELECT * FROM Employees WHERE FirstName = 'ali';
```

**Does this return one row, or three?** The answer isn't in the query — it's determined entirely by a setting sitting quietly underneath the column: **Collation**. Get it wrong (or never think about it) and a `WHERE`, `JOIN`, or `ORDER BY` can silently behave in a way that looks like a bug but is actually working exactly as configured.

In [ ]:
-- Run this in SSMS
CREATE TABLE #Employees (FirstName NVARCHAR(50));
INSERT INTO #Employees VALUES (N'Ali'), (N'ali'), (N'ALI');

-- Default database collation decides the result here
SELECT * FROM #Employees WHERE FirstName = 'ali';

DROP TABLE #Employees;

## 2. What Is Collation? (Plain Definition)

**Collation is a property of character data that defines the rules for comparing and sorting it.** A collation name like `Latin1_General_100_CI_AS` packs several of these rules together into one identifier — which alphabet's rules to use, which version of those rules, and several yes/no sensitivity flags, covered next.

## 3. The Full Set of Collation Options — With a Runnable Example for Each

| Suffix | Full Name | What It Controls |
|---|---|---|
| `CS` / `CI` | Case-sensitive / Case-insensitive | Whether `A` and `a` are treated as equal |
| `AS` / `AI` | Accent-sensitive / Accent-insensitive | Whether `e` and `é` are treated as equal |
| `KS` | Kana-sensitive | Distinguishes Japanese Hiragana from Katakana (no "KI" -- omitted means always equal) |
| `WS` | Width-sensitive | Distinguishes full-width from half-width characters (no "WI") |
| `VSS` | Variation-selector-sensitive | Distinguishes characters differing only by a Unicode variation selector |
| `SC` | Supplementary Character support | Correctly handles Unicode characters needing more than 2 bytes |
| `BIN` | Binary | Raw byte-value comparison -- fastest, ignores all linguistic rules |
| `BIN2` | Binary Code Point | More precise binary comparison, based on Unicode code points |

**Important interaction:** `BIN`/`BIN2` are mutually exclusive with every other option above -- a binary collation skips linguistic interpretation entirely.

### CS vs. CI -- Case Sensitivity

SELECT
    CASE WHEN 'Ali' = 'ali' COLLATE Latin1_General_CI_AS THEN 'Equal' ELSE 'Not Equal' END AS CI_Result,
    CASE WHEN 'Ali' = 'ali' COLLATE Latin1_General_CS_AS THEN 'Equal' ELSE 'Not Equal' END AS CS_Result;

-- Expected output:
-- CI_Result   CS_Result
-- Equal       Not Equal

### AS vs. AI -- Accent Sensitivity

In [ ]:
SELECT
    CASE WHEN N'e' = N'é' COLLATE Latin1_General_CI_AI THEN 'Equal' ELSE 'Not Equal' END AS AI_Result,
    CASE WHEN N'e' = N'é' COLLATE Latin1_General_CI_AS THEN 'Equal' ELSE 'Not Equal' END AS AS_Result;

-- Expected output:
-- AI_Result   AS_Result
-- Equal       Not Equal

### KS -- Kana Sensitivity (Japanese Hiragana vs. Katakana)

Using あ (Hiragana "a", U+3042) and ア (Katakana "a", U+30A2) -- the same sound, two different scripts.

In [ ]:
SELECT
    CASE WHEN N'あ' = N'ア' COLLATE Japanese_CI_AS THEN 'Equal' ELSE 'Not Equal' END AS KS_Not_Specified,
    CASE WHEN N'あ' = N'ア' COLLATE Japanese_CI_AS_KS THEN 'Equal' ELSE 'Not Equal' END AS KS_Specified;

-- Expected output:
-- KS_Not_Specified   KS_Specified
-- Equal              Not Equal

### WS -- Width Sensitivity (Full-width vs. Half-width)

Using `A` (regular, half-width) and `Ａ` (full-width Latin "A", U+FF21) -- visually similar, different code points.

In [ ]:
SELECT
    CASE WHEN N'A' = N'Ａ' COLLATE Japanese_CI_AS THEN 'Equal' ELSE 'Not Equal' END AS WS_Not_Specified,
    CASE WHEN N'A' = N'Ａ' COLLATE Japanese_CI_AS_WS THEN 'Equal' ELSE 'Not Equal' END AS WS_Specified;

-- Expected output:
-- WS_Not_Specified   WS_Specified
-- Equal              Not Equal

### VSS -- Variation-Selector Sensitivity (SQL Server 2017+, on supporting collations)

Some CJK characters have a "base" form and a form followed by an invisible Unicode variation selector (U+FE00-U+FE0F) that requests a specific visual style. `VSS` controls whether these count as different characters. This is a narrower, newer feature -- supported only on certain `_140` collations.

In [ ]:
-- Base character vs. the same character + variation selector U+FE00
SELECT
    CASE WHEN N'邊' = N'邊︀' COLLATE Japanese_XJIS_140_CI_AS THEN 'Equal' ELSE 'Not Equal' END AS VSS_Not_Specified,
    CASE WHEN N'邊' = N'邊︀' COLLATE Japanese_XJIS_140_CI_AS_VSS THEN 'Equal' ELSE 'Not Equal' END AS VSS_Specified;

-- Expected output:
-- VSS_Not_Specified   VSS_Specified
-- Equal               Not Equal

### SC -- Supplementary Character Support

Some Unicode characters (rare CJK ideographs, many emoji) live outside the Basic Multilingual Plane and need a surrogate pair (2 UTF-16 code units) to represent -- these are "supplementary characters." `SC`-enabled `_100`+ collations recognize such a pair as **one character**, not two, which affects `LEN()` and sorting correctness.

In [ ]:
-- U+20000 is a supplementary-plane CJK character, stored as a surrogate pair
SELECT LEN(N'𠀀') AS Length_With_SC_Collation;
-- On a database/column with an SC-enabled collation: returns 1 (one real character)
-- On a legacy non-SC collation: may be treated as 2 code units instead of 1 character

### BIN vs. BIN2 -- Binary Comparisons

Both skip linguistic rules entirely and compare raw values -- which also makes case-sensitivity automatic (there's no "insensitive" binary collation).

In [ ]:
SELECT
    CASE WHEN 'a' = 'A' COLLATE Latin1_General_BIN THEN 'Equal' ELSE 'Not Equal' END AS BIN_Result,
    CASE WHEN 'a' = 'A' COLLATE Latin1_General_BIN2 THEN 'Equal' ELSE 'Not Equal' END AS BIN2_Result;

-- Expected output:
-- BIN_Result   BIN2_Result
-- Not Equal    Not Equal
-- (Both are inherently case-sensitive; BIN vs BIN2 differ mainly in sort ORDER
--  for certain multi-byte characters, not in simple equality checks like this one.)

## 4. Three Collation Families in SQL Server

| Family | Description |
|---|---|
| **Windows Collations** | Based on Windows OS locale rules -- the modern, recommended default for new development (e.g., `Latin1_General_100_CI_AS`, `Persian_100_CI_AS`) |
| **SQL Server Collations** | Older, SQL-Server-specific rules, names start with `SQL_` (e.g., `SQL_Latin1_General_CP1_CI_AS`) -- kept mainly for backward compatibility |
| **Binary Collations** | Raw value comparison (`BIN`/`BIN2`) -- no linguistic rules applied at all |

## 5. Code Pages -- Relevant Only for Non-Unicode Types

`CHAR`/`VARCHAR`/`TEXT` store exactly one byte per character, so only 256 characters can exist at once -- collation determines *which* 256-character set (the **code page**) is in use. A Hebrew collation and an English collation on a `VARCHAR` column can map the same raw byte to a completely different displayed character.

`NCHAR`/`NVARCHAR`/`NTEXT` (Unicode types) sidestep this entirely, since they reserve enough space per character to avoid needing a restrictive code page -- which is exactly why multilingual columns should default to `NVARCHAR`.

## 6. The Four Collation Levels -- With T-SQL for Each

| Level | Scope |
|---|---|
| **Server / Instance** | Set once at installation; default for system databases and new user databases |
| **Database** | Default for everything inside that database |
| **Column** | Overrides the database default for one column |
| **Expression** | A one-query, temporary override |

Each level overrides the one above it when explicitly specified.

In [ ]:
-- 1. Server-level: check the instance's default collation
SELECT SERVERPROPERTY('collation');

-- 2. Database-level: set at creation, or change afterward
CREATE DATABASE GeekDB
COLLATE Greek_CS_AI;

ALTER DATABASE GeekDB
COLLATE SQL_Latin1_General_CP1_CI_AS;

-- 3. Column-level: overrides the database default for one column
ALTER TABLE Geektable
ALTER COLUMN namecol NVARCHAR(10) COLLATE Greek_CS_AI;

-- 4. Expression-level: a temporary, query-scoped override
SELECT * FROM Geektab
ORDER BY columnname COLLATE SQL_Latin1_General_CP1_CI_AS;

### The Practical Reason Expression-Level Matters: Collation Conflicts

Joining two tables from databases with *different* collations throws an error like:

```text
Cannot resolve the collation conflict between "Latin1_General_100_CI_AS"
and "SQL_Latin1_General_CP1_CI_AS" in the equal to operation.
```

Expression-level `COLLATE` fixes this without touching the schema -- just force both sides to agree for that one query:

In [ ]:
SELECT a.CustomerName
FROM DatabaseA.dbo.Customer a
JOIN DatabaseB.dbo.[Order] b
    ON a.CustomerCode = b.CustomerCode COLLATE Latin1_General_100_CI_AS;

## 7. Diagnostic Queries Worth Knowing

In [ ]:
-- What's the current server's default collation?
SELECT SERVERPROPERTY('collation');

-- What collations are actually available on this instance?
SELECT * FROM sys.fn_helpcollations();

-- Search specifically for Persian-related collations
SELECT * FROM sys.fn_helpcollations()
WHERE name LIKE 'Persian%';

`sys.fn_helpcollations()` returns every supported collation with a plain-language description -- the last query above is the fastest way to see every Persian-related option actually available on your instance.

## 8. Applying This: What to Choose for Persian (Farsi) Text

SQL Server has dedicated Persian collations, part of the Windows family, `_100` version:

```text
Persian_100_CI_AS
Persian_100_CI_AI
Persian_100_CS_AS
Persian_100_BIN2
```

**Recommended default:**

In [ ]:
CREATE DATABASE SematecLMS
COLLATE Persian_100_CI_AS;

USE SematecLMS;

CREATE TABLE Employee (
    EmployeeCode INT IDENTITY(1,1) PRIMARY KEY,
    FirstName NVARCHAR(50) COLLATE Persian_100_CI_AS,
    LastName  NVARCHAR(50) COLLATE Persian_100_CI_AS
);

INSERT INTO Employee (FirstName, LastName) VALUES
    (N'علی', N'رحمانی'),
    (N'فاطمه', N'رضایی');

-- Correct Persian sort order, including letters like p ch zh g that
-- generic Arabic collations sort incorrectly
SELECT * FROM Employee ORDER BY FirstName;

**Why `Persian_100_CI_AS`, broken down:**

- **`Persian`** -- correct Persian alphabetical sort order. This matters concretely: generic Arabic collations sort Persian-only letters (`پ`, `چ`, `ژ`, `گ`) incorrectly, since those letters don't exist in Arabic at all.
- **`100`** -- the modern collation version, with better Unicode handling than the legacy default.
- **`CI`** -- Persian script itself has no case distinction, but this still matters for any English text mixed into the same database, which is the common case.
- **`AS`** -- relevant if the same system also stores other accented Latin text (French, German) or needs precise diacritic handling.

### Two Requirements That Must Go Together With This

1. **Always use `NVARCHAR`/`NCHAR`, never `VARCHAR`/`CHAR`, for Persian text.** Non-Unicode types depend on a code page, and Persian characters won't display correctly under most of them -- a data type decision, entirely separate from collation, and both need to be correct together.
2. **Keep the collation consistent across the whole system** -- database, all relevant columns, and any linked databases. Older client drivers not recognizing a newer collation version (`Persian_100_CI_AI`) is a documented real-world failure mode -- worth testing actual client tools, not just SSMS, against whichever collation you choose.

### If Exact, Case-and-Accent-Precise Matching Is Needed

In [ ]:
-- Rare for Persian specifically, but relevant for exact document/legal-text matching
ALTER TABLE Employee
ALTER COLUMN FirstName NVARCHAR(50) COLLATE Persian_100_CS_AS;

### Mixed Persian + English Systems (the Common Case)

A single database-level `Persian_100_CI_AS` collation generally handles both scripts correctly within `NVARCHAR` columns -- no need for a different collation per column just because content is bilingual. Persian's sort rules fall back gracefully to standard ordering for the Latin characters they don't specifically govern.

## 9. Quick Reference (Scan This Next Time You Forget)

| Question | Answer |
|---|---|
| What does collation control? | How character data is **compared** and **sorted** |
| CI vs CS | Case-insensitive (A=a) vs. case-sensitive (A≠a) |
| AI vs AS | Accent-insensitive (e=é) vs. accent-sensitive (e≠é) |
| KS | Distinguishes Hiragana from Katakana in Japanese |
| WS | Distinguishes full-width from half-width characters |
| VSS | Distinguishes characters differing by a variation selector |
| SC | Correctly counts/handles supplementary (surrogate-pair) characters |
| BIN / BIN2 | Raw-value comparison, no linguistic rules, mutually exclusive with CI/CS/AI/AS |
| 4 levels, in override order | Server -> Database -> Column -> Expression (most specific wins) |
| Check server default | `SELECT SERVERPROPERTY('collation');` |
| List all available collations | `SELECT * FROM sys.fn_helpcollations();` |
| Fix a JOIN collation conflict | `ON a.Col = b.Col COLLATE <matching_collation>` |
| Persian text -- collation | `Persian_100_CI_AS` (database + column level) |
| Persian text -- data type | `NVARCHAR` / `NCHAR`, never `VARCHAR`/`CHAR` |

##  Key Takeaways

- Collation isn't a display-language setting -- it's the rulebook for **comparing and sorting** text, operating silently under every `WHERE`, `JOIN`, and `ORDER BY`.
- Beyond `CI`/`CS`/`AI`/`AS`, SQL Server also supports `KS`, `WS`, `VSS`, `SC`, and binary (`BIN`/`BIN2`) options -- each solving a narrower, specific comparison problem, each demonstrated above with a real, runnable query.
- Collation applies at **four** levels -- Server, Database, Column, Expression -- with the most specific one always winning, and Expression-level being the lightweight fix for cross-database collation conflicts.
- For Persian text specifically: pair a `Persian_100_*` collation (commonly `Persian_100_CI_AS`) with `NVARCHAR`/`NCHAR` columns -- collation fixes sort order, Unicode data types fix storage, and both are required together.